# Customer Behaviour Analysis with Spark

An Apache Spark analysis covering transaction preparation, joins, cleaning, exploratory analysis, customer segmentation, and RFM analysis. 

This educational project includes starter material and exercises. See the README for the data requirements. Results should be reproduced with the appropriate dataset.


# Customer Behaviour Analysis

# Objective
In this case study, you will be working on E-commerce Customer Behavior Analysis using Apache Spark, a powerful distributed computing framework designed for big data processing. This assignment aims to give you hands-on experience in analyzing large-scale e-commerce datasets using PySpark. You will apply techniques learned in data analytics to clean, transform, and explore customer behavior data, drawing meaningful insights to support business decision-making. Apart from understanding how big data tools can optimize performance on a single machine and across clusters, you will develop a structured approach to analyzing customer segmentation, purchase patterns, and behavioral trends.

# Business Value
E-commerce businesses operate in a highly competitive market where understanding customer behavior is critical to driving growth and retention. To stay ahead, companies must leverage data-driven insights to optimize marketing strategies, personalize customer experiences, and improve product offerings. In this assignment, you will analyze e-commerce transaction data to uncover patterns in purchasing behavior, customer preferences, and sales performance. With Apache Spark's ability to handle large datasets efficiently, businesses can process vast amounts of customer interactions in real-time, helping them make faster and more informed decisions.
As an analyst at an e-commerce company, your task is to examine historical transaction records and customer survey data to derive actionable insights that can drive business growth. Your analysis will help identify high-value customers, segment users based on behavior, and uncover trends in product demand and customer engagement. By leveraging big data analytics, businesses can enhance customer satisfaction, improve retention rates, and maximize revenue opportunities.


# Assignment Tasks
1. Data Preparation
2. Data Cleaning
3. Exploratory Data Analysis
4. Customer Segmentation (RFM Analysis) and Business Insights
5. Evaluation and Conclusion


# Dataset Overview
The dataset can be accessed the following [link](DATA_SOURCE_NOT_REDISTRIBUTED).

The dataset used in this analysis comprises longitudinal purchase records from 5,027 Amazon.com users in the United States, spanning 2018 to 2022.

It is structured into three CSV files (amazon-purchases.csv, survey.csv, and fields.csv) that capture transactional data, demographic profiles, and survey responses.

Collected with informed consent, the dataset enables analysis of customer behavior, product preferences, and demographic trends.

**NOTE**: Personal identifiers (PII) were removed to ensure privacy, and all data were preprocessed by users before submission.

`Data Dictionary:`

| **Attribute**          | **Description** |
|------------------------|----------------|
| **Order Dates**        | The specific dates when orders were placed, enabling chronological analysis of sales trends. |
| **Title** |The name of the product purchased. |
|**Category** | The classification or group to which the product belongs, facilitating category-wise analysis. |
| **Pricing** | The cost per unit of each product, essential for revenue calculations and pricing strategy assessments. |
| **Quantities** | The number of units of each product ordered in a transaction, aiding in inventory and demand analysis. |
| **Shipping States**    | The states to which products were shipped, useful for geographical sales distribution analysis. |
| **Survey ResponseID**  | A unique identifier linking purchases to customer survey responses, enabling correlation between purchasing behavior and customer feedback. |



# Loading the Datasets

In [ ]:
from pyspark.sql import SparkSession

In [ ]:
spark = SparkSession.builder.appName('Danish').getOrCreate()

In [ ]:
sc

In [ ]:
print("Spark version:", spark.version)

In [ ]:
for package in [
    "Pillow",
    "contourpy",
    "cycler",
    "fonttools",
    "importlib-resources",
    "kiwisolver"
]:
    sc.install_pypi_package(package)

In [ ]:
import matplotlib
print(matplotlib.__version__)

import matplotlib.pyplot as plt
import seaborn as sns

print("Plotting libraries loaded successfully")

In [ ]:
from pathlib import Path
# Import necessary libraries
from pyspark.sql import SparkSession

# Initialise Spark session
spark = SparkSession.builder \
    .appName("Customer Behavior Analysis") \
    .getOrCreate()


# Load the paths to the datasets/csv files
BASE_PATH = str(Path("data").resolve())
amazon_purchases_path = f"{BASE_PATH}/amazon-purchases.csv"
survey_path = f"{BASE_PATH}/survey.csv"
fields_path = f"{BASE_PATH}/fields.csv"


# Load datasets into PySpark DataFrames
amazon_purchases = spark.read.csv(amazon_purchases_path, header=True, inferSchema=True)
survey = spark.read.csv(survey_path, header=True, inferSchema=True)
fields = spark.read.csv(fields_path, header=True, inferSchema=True)

# Merge the datasets
merged_data = amazon_purchases.join(
    survey,
    on="Survey ResponseID",
    how="left"
)

# Display the merged data
#merged_data.show(5, truncate=30, vertical=True)
merged_data_sample = merged_data.limit(5)

In [ ]:
print("amazon_purchases_Rows:", amazon_purchases.count())
print("amazon_purchases_Columns:", len(amazon_purchases.columns))
print("survey_Rows:", survey.count())
print("survey_Columns:", len(survey.columns))
print("fields_Rows:", fields.count())
print("fields_Columns:", len(fields.columns))

In [ ]:
merged_data_sample

In [ ]:
amazon_purchases.toPandas().info()

#1. Data Preparation

Before analysis, the data needs to be prepared to ensure consistency and efficiency.
- Check for data consistency and ensure all columns are correctly formatted.
- Structure and prepare the dataset for further processing, ensuring that relevant features are retained


In [ ]:
from pyspark.sql.functions import sum as spark_sum, col

# Check for missing values in the merged dataset
missing_values = merged_data.select([
    spark_sum(col(c).isNull().cast("int")).alias(c)
    for c in merged_data.columns
])

#missing_values.show(truncate=False, vertical=True)

In [ ]:
missing_values

#2. Data Cleaning <font color = red>[20 marks]</font> <br>

Prepare the data for further analysis by performing data cleaning such as missing value treatment, handle data schema, outlier analysis, and relevant feature engineering techniques.

## 2.1 Handling Missing values <font color = red>[10 marks]</font> <br>
Handle missing values in the data

In [ ]:
# Import necessary functions
from pyspark.sql.functions import when, col, sum as spark_sum

# Fill missing (null) values with the appropriate techniques as required by the analysis
numeric_columns = [
    column_name
    for column_name, data_type in merged_data.dtypes
    if data_type in ["int", "bigint", "float", "double"]
]

categorical_columns = [
    column_name
    for column_name, data_type in merged_data.dtypes
    if data_type == "string"
]

merged_data = merged_data.fillna(0, subset=numeric_columns)
merged_data = merged_data.fillna("Unknown", subset=categorical_columns)

# Aggregate and count missing values (nulls) for each column after replacement
missing_values_after = merged_data.select([
    spark_sum(col(c).isNull().cast("int")).alias(c)
    for c in merged_data.columns
])

# Display the count of remaining missing values in each column
#missing_values_after.show(truncate=False, vertical=True)

In [ ]:
missing_values_after

## 2.2 Feature Engineering <font color = red>[5 marks]</font> <br>
Perform feature engineering on the dataset to extract relevant/ create new features as required and map specific data types.

In [ ]:
from pyspark.sql.functions import col, month, year, to_date

# Perform appropriate feature engineering. Eg. Extract order date, month, year and cast to the appropriate values
merged_data = (
    merged_data
    .withColumn("Order Date", to_date(col("Order Date"), "yyyy-MM-dd"))
    .withColumn("Purchase Price Per Unit", col("Purchase Price Per Unit").cast("double"))
    .withColumn("Quantity", col("Quantity").cast("integer"))
    .withColumn("Order Month", month(col("Order Date")))
    .withColumn("Order Year", year(col("Order Date")))
    .withColumn(
        "Total Amount",
        col("Purchase Price Per Unit") * col("Quantity")
    )
)

# Display the updated dataset
#merged_data.show(10, truncate=30, vertical=True)
merged_data_sample = merged_data.limit(5)

In [ ]:
merged_data_sample

In [ ]:
from pyspark.sql.functions import create_map, lit
from itertools import chain

# Map categorical income to numerical values
income_mapping = {
    'Less than $25,000': 0,
    '$25,000 - $49,999': 1,
    '$50,000 - $74,999': 2,
    '$75,000 - $99,999': 3,
    '$100,000 - $149,999': 4,
    '$150,000 or more': 5
}

income_map = create_map(
    *[lit(value) for value in chain(*income_mapping.items())]
)

merged_data = merged_data.withColumn(
    "Income Numeric",
    income_map[col("Q-demos-income")]
)

# Map gender to numerical values
gender_mapping = {
    'Female': 0,
    'Male': 1,
    'Other': 2,
    'Prefer not to say': -1,
    'Unknown': -1
}

gender_map = create_map(
    *[lit(value) for value in chain(*gender_mapping.items())]
)

merged_data = merged_data.withColumn(
    "Gender Numeric",
    gender_map[col("Q-demos-gender")]
)


# Display the updated dataset
merged_data.select(
    "Q-demos-income",
    "Income Numeric",
    "Q-demos-gender",
    "Gender Numeric"
).show(10, truncate=False)

## 2.3 Data Cleaning <font color = red>[5 marks]</font> <br>
Handle data cleaning techniques such as data duplication, dropping unnecessary values etc.

In [ ]:
# Check for duplicates
print("Number of Duplicates:", merged_data.count() - merged_data.dropDuplicates().count())

# Remove duplicates
merged_data = merged_data.dropDuplicates()

# Verify duplicates after cleaning
print("Number of Duplicates After Cleaning:", merged_data.count() - merged_data.dropDuplicates().count())

# Enrich missing product categories using product code and title

from pyspark.sql.functions import (
    col,
    count as spark_count,
    row_number,
    when,
    coalesce,
    lit
)
from pyspark.sql.window import Window

# Select records with known categories
known_categories = merged_data.filter(
    col("Category").isNotNull() &
    (col("Category") != "Unknown")
)

# Create product-code-to-category dimension table
asin_category_counts = (
    known_categories
    .filter(
        col("ASIN/ISBN (Product Code)").isNotNull() &
        (col("ASIN/ISBN (Product Code)") != "Unknown")
    )
    .groupBy(
        "ASIN/ISBN (Product Code)",
        "Category"
    )
    .agg(
        spark_count("*").alias("Category Count")
    )
)

asin_window = Window.partitionBy(
    "ASIN/ISBN (Product Code)"
).orderBy(
    col("Category Count").desc(),
    col("Category")
)

asin_category_dimension = (
    asin_category_counts
    .withColumn(
        "Category Rank",
        row_number().over(asin_window)
    )
    .filter(col("Category Rank") == 1)
    .select(
        col("ASIN/ISBN (Product Code)").alias("Product Code Key"),
        col("Category").alias("ASIN Category")
    )
)

# Create product-title-to-category dimension table
title_category_counts = (
    known_categories
    .filter(
        col("Title").isNotNull() &
        (col("Title") != "Unknown")
    )
    .groupBy(
        "Title",
        "Category"
    )
    .agg(
        spark_count("*").alias("Category Count")
    )
)

title_window = Window.partitionBy("Title").orderBy(
    col("Category Count").desc(),
    col("Category")
)

title_category_dimension = (
    title_category_counts
    .withColumn(
        "Category Rank",
        row_number().over(title_window)
    )
    .filter(col("Category Rank") == 1)
    .select(
        col("Title").alias("Title Key"),
        col("Category").alias("Title Category")
    )
)

# Join category dimension tables to the transaction data
merged_data = (
    merged_data.alias("p")
    .join(
        asin_category_dimension.alias("a"),
        col("p.ASIN/ISBN (Product Code)") ==
        col("a.Product Code Key"),
        how="left"
    )
    .join(
        title_category_dimension.alias("t"),
        col("p.Title") == col("t.Title Key"),
        how="left"
    )
    .select(
        "p.*",
        col("a.ASIN Category"),
        col("t.Title Category")
    )
)

# Replace unknown categories using ASIN first and title second
merged_data = (
    merged_data
    .withColumn(
        "Category",
        when(
            col("Category").isNull() |
            (col("Category") == "Unknown"),
            coalesce(
                col("ASIN Category"),
                col("Title Category"),
                lit("Unknown")
            )
        ).otherwise(col("Category"))
    )
    .drop(
        "ASIN Category",
        "Title Category"
    )
)

# Display remaining unknown category count
print("Remaining Unknown Categories:")

merged_data.filter(
    col("Category") == "Unknown"
).groupBy("Category").count().show()

In [ ]:
cleaned_data_path = f"{BASE_PATH}/cleaned_data" # Save the cleaned dataset locally
merged_data.write.csv(cleaned_data_path, header=True, mode='overwrite')

# Load the cleaned dataset from the location
cleaned_data = spark.read.csv(cleaned_data_path, header=True, inferSchema=True)

# Display the first few rows
#print("Cleaned Data:")
#cleaned_data.show(5, truncate=30, vertical=True)
cleaned_data_sample = cleaned_data.limit(5)

In [ ]:
cleaned_data_sample

# 3. Exploratory Data Analysis <font color = red>[55 marks]</font> <br>

## 3.1 Analyse purchases by hour, day and month <font color = red>[5 marks]</font> <br>

Examine overall trends in purchases over time and analyse the trends by hour, day, month.

In [ ]:
# Purchase Distribution by Hour, Day, and Month

from pyspark.sql.functions import hour, dayofweek, month
import seaborn as sns
import matplotlib.pyplot as plt

# Extract hour, day, and month
purchase_time = (
    cleaned_data
    .withColumn("Day", dayofweek(col("Order Date")))
    .withColumn("Month", month(col("Order Date")))
)

# Group and count purchases by time factors
day_counts = (
    purchase_time
    .groupBy("Day")
    .count()
    .orderBy("Day")
)

month_counts = (
    purchase_time
    .groupBy("Month")
    .count()
    .orderBy("Month")
)

# Convert to Pandas for visualisation
day_pd = day_counts.toPandas()
month_pd = month_counts.toPandas()

day_names = {
    1: "Sunday",
    2: "Monday",
    3: "Tuesday",
    4: "Wednesday",
    5: "Thursday",
    6: "Friday",
    7: "Saturday"
}

day_pd["Day Name"] = day_pd["Day"].map(day_names)


# Plot the data
fig, axes = plt.subplots(1, 3, figsize=(18, 5))
axes[0].text(
    0.5,
    0.5,
    "Hourly analysis unavailable\n(Order Date has no time component)",
    ha="center",
    va="center"
)
axes[0].set_title("Purchases by Hour")
axes[0].set_axis_off()

sns.barplot(data=day_pd, x="Day Name", y="count", ax=axes[1])
axes[1].set_title("Purchases by Day")
axes[1].set_xlabel("Day")
axes[1].set_ylabel("Number of Purchases")
axes[1].tick_params(axis="x", rotation=45)

sns.barplot(data=month_pd, x="Month", y="count", ax=axes[2])
axes[2].set_title("Purchases by Month")
axes[2].set_xlabel("Month")
axes[2].set_ylabel("Number of Purchases")

plt.tight_layout()
plt.show()

In [ ]:
# Monthly Purchase Trends

from pyspark.sql.functions import date_format, col
import matplotlib.pyplot as plt
import seaborn as sns

# Extract month and year from 'Order Date'
monthly_data = cleaned_data.withColumn(
    "Month-Year",
    date_format(col("Order Date"), "yyyy-MM")
)

# Group by month and count purchases
monthly_counts = (
    monthly_data
    .groupBy("Month-Year")
    .count()
    .orderBy("Month-Year")
)

# Convert to Pandas for visualisation
monthly_pd = monthly_counts.toPandas()



# Plot
plt.figure(figsize=(14, 6))

sns.lineplot(
    data=monthly_pd,
    x="Month-Year",
    y="count",
    marker="o"
)

plt.title("Monthly Purchase Trends")
plt.xlabel("Month and Year")
plt.ylabel("Number of Purchases")
plt.xticks(rotation=90)
plt.tight_layout()
plt.show()

In [ ]:
# Yealy Purchase Trends

from pyspark.sql.functions import date_format, col
import matplotlib.pyplot as plt
import seaborn as sns


# Group by Year and count purchases
yearly_counts = (
    cleaned_data
    .withColumn("Year", date_format(col("Order Date"), "yyyy"))
    .groupBy("Year")
    .count()
    .orderBy("Year")
)

# Convert to Pandas for visualisation
yearly_pd = yearly_counts.toPandas()

# Plot
plt.figure(figsize=(10, 5))

sns.barplot(
    data=yearly_pd,
    x="Year",
    y="count"
)
plt.title("Yearly Purchase Trends")
plt.xlabel("Year")
plt.ylabel("Number of Purchases")
plt.tight_layout()
plt.show()

## 3.2 Customer Demographics vs Purchase Frequency <font color = red>[5 marks]</font> <br>
Analyse the trends between the customer deographics and the purchase frequency

In [ ]:
# Correlation Between Demographics and Purchase Frequency

from pyspark.sql.functions import count

# Group by demographic attributes and count purchases
demographic_purchase_frequency = (
    cleaned_data
    .groupBy("Q-demos-age", "Q-demos-gender")
    .agg(count("*").alias("Purchase Frequency"))
)

# Convert to Pandas for visualisation
demographic_pd = demographic_purchase_frequency.toPandas()

# Plot
plt.figure(figsize=(12, 6))

sns.barplot(
    data=demographic_pd,
    x="Q-demos-age",
    y="Purchase Frequency",
    hue="Q-demos-gender"
)

plt.title("Customer Demographics vs Purchase Frequency")
plt.xlabel("Age Group")
plt.ylabel("Purchase Frequency")
plt.xticks(rotation=45)
plt.tight_layout()
plt.show()

## 3.3 Purchase behavior weekend vs weekday <font color = red>[5 marks]</font> <br>

Compare the purchase behavior of customer's on weekdays vs. weekends.

In [ ]:
# Weekday vs. Weekend Purchase Behavior

from pyspark.sql.functions import when, dayofweek, col

# Define weekdays and weekends
weekday_weekend_data = cleaned_data.withColumn(
    "Day Type",
    when(
        dayofweek(col("Order Date")).isin(1, 7),
        "Weekend"
    ).otherwise("Weekday")
)

# Group and count purchases
weekday_weekend_counts = (
    weekday_weekend_data
    .groupBy("Day Type")
    .count()
)

# Convert to Pandas for visualisation
weekday_weekend_pd = weekday_weekend_counts.toPandas()

# Plot
plt.figure(figsize=(7, 5))

sns.barplot(
    data=weekday_weekend_pd,
    x="Day Type",
    y="count"
)

plt.title("Weekday vs. Weekend Purchase Behavior")
plt.xlabel("Day Type")
plt.ylabel("Number of Purchases")
plt.tight_layout()
plt.show()

## 3.4 Frequently purchased product pairs <font color = red>[5 marks]</font> <br>

Analyze how frequently products are purchased together (also known as Market Basket Analysis)


In [ ]:
# Weekday vs. Weekend Purchase Behavior

from pyspark.sql.functions import when, dayofweek, col

# Define weekdays and weekends
weekday_weekend_data = cleaned_data.withColumn(
    "Day Type",
    when(
        dayofweek(col("Order Date")).isin(1, 7),
        "Weekend"
    ).otherwise("Weekday")
)

# Group and count purchases
weekday_weekend_counts = (
    weekday_weekend_data
    .groupBy("Day Type")
    .count()
)

# Convert to Pandas for visualisation
weekday_weekend_pd = weekday_weekend_counts.toPandas()

# Plot
plt.figure(figsize=(7, 5))

sns.barplot(
    data=weekday_weekend_pd,
    x="Day Type",
    y="count"
)

plt.title("Weekday vs. Weekend Purchase Behavior")
plt.xlabel("Day Type")
plt.ylabel("Number of Purchases")
plt.tight_layout()
plt.show()

## 3.5 Examine Product Performance <font color = red>[5 marks]</font> <br>

Examine the performance of products by calculating revenue and item popularity.

In [ ]:
# Contribution of Product Categories (Top 25)
from pyspark.sql.functions import sum as spark_sum, col

# Rename categories that could not be recovered
cleaned_data = cleaned_data.withColumn(
    "Category",
    when(
        col("Category") == "Unknown",
        "UNCLASSIFIED"
    ).otherwise(col("Category"))
)

# Group by category and sum 'Total Amount' to get total revenue
category_contribution = (
    cleaned_data
    .groupBy("Category")
    .agg(
        spark_sum("Total Amount").alias("Total Revenue"),
        spark_sum("Quantity").alias("Item Popularity")
    )
    .orderBy(col("Total Revenue").desc())
    .limit(25)
)


# Convert to Pandas for visualisation
category_pd = category_contribution.toPandas()

# Plot
plt.figure(figsize=(12, 8))

sns.barplot(
    data=category_pd,
    x="Total Revenue",
    y="Category"
)

plt.title("Contribution of Product Categories (Top 25)")
plt.xlabel("Total Revenue")
plt.ylabel("Product Category")
plt.tight_layout()
plt.show()

## 3.6 Top products by quantity <font color = red>[5 marks]</font> <br>

Identify the most frequently purchased products.

In [ ]:
# Top 10 Products by Quantity
from pyspark.sql.functions import sum as spark_sum, col, first

# Group by product title and sum 'Quantity'
top_products = (
    cleaned_data
    .groupBy("ASIN/ISBN (Product Code)")
    .agg(
        spark_sum("Quantity").alias("Total Quantity")
    )
    .orderBy(col("Total Quantity").desc())
    .limit(10)
)

# Convert to Pandas for visualisation
top_products_pd = top_products.toPandas()


# Plot
plt.figure(figsize=(12, 7))

sns.barplot(
    data=top_products_pd,
    x="Total Quantity",
    y="ASIN/ISBN (Product Code)"
)

plt.title("Top 10 Products by Quantity")
plt.xlabel("Total Quantity")
plt.ylabel("Product Code")
plt.tight_layout()

plt.show()

## 3.7 Distribution of Purchases by State <font color = red>[5 marks]</font> <br>

Analyze the distribution of purchases across states and categories.

In [ ]:
# Distribution of Purchases by State (Top 25)
from pyspark.sql.functions import col

state_distribution = (
    cleaned_data
    .groupBy("Shipping Address State")
    .count()
    .orderBy(col("count").desc())
    .limit(25)
)

# Convert to Pandas for visualisation
state_pd = state_distribution.toPandas()

# Plot
plt.figure(figsize=(12, 7))

sns.barplot(
    data=state_pd,
    x="count",
    y="Shipping Address State"
)

plt.title("Distribution of Purchases by State (Top 25)")
plt.xlabel("Number of Purchases")
plt.ylabel("Shipping State")
plt.tight_layout()
plt.show()

## 3.8 Price vs Product Quantity <font color = red>[5 marks]</font> <br>

Identify the Relationship between Price and Quantity

In [ ]:
# Relationship Between Price and Quantity
from pyspark.sql.functions import col
import pandas as pd

price_quantity = (
    cleaned_data
    .select(
        col("Purchase Price Per Unit")
        .cast("double")
        .alias("Purchase Price Per Unit"),

        col("Quantity")
        .cast("double")
        .alias("Quantity")
    )
    .filter(
        (col("Purchase Price Per Unit") > 0) &
        (col("Quantity") > 0)
    )
)

# Calculate correlation using the complete Spark dataset
price_quantity_correlation = price_quantity.stat.corr(
    "Purchase Price Per Unit",
    "Quantity"
)

print(
    "Correlation between Price and Quantity:",
    round(price_quantity_correlation, 4)
)


# Convert to Pandas for visualisation
price_quantity_pd = (
    price_quantity
    .sample(
        withReplacement=False,
        fraction=0.01,
        seed=42
    )
    .limit(10000)
    .toPandas()
)

price_quantity_pd["Purchase Price Per Unit"] = pd.to_numeric(
    price_quantity_pd["Purchase Price Per Unit"]
)

price_quantity_pd["Quantity"] = pd.to_numeric(
    price_quantity_pd["Quantity"]
)

# Plot
plt.figure(figsize=(10, 6))

ax = sns.scatterplot(
    data=price_quantity_pd,
    x="Purchase Price Per Unit",
    y="Quantity",
    alpha=0.4
)

ax.set_yticks([1, 2, 3, 5, 10, 15, 20])
ax.set_yticklabels([
    "1 (Single item)",
    "2 items",
    "3 items",
    "5 items",
    "10 items",
    "15 items",
    "20 items"
])

plt.title("Relationship Between Price and Quantity")
plt.xlabel("Purchase Price Per Unit ($)")
plt.ylabel("Quantity Purchased")
plt.tight_layout()

plt.show()

## 3.9 Analyse the spending KPIs <font color = red>[5 marks]</font> <br>


A popular KPI is average spend per customer. Calculate this metric as the ratio of total transaction amount from non-recurring payments divided by the total number of customers who made a purchase

In [ ]:
# Average Spend per Customer

from pyspark.sql.functions import avg, sum as spark_sum

# Group by customer and calculate average spend
customer_spending = (
    cleaned_data
    .filter(
        col("Survey ResponseID").isNotNull() &
        (col("Survey ResponseID") != "Unknown")
    )
    .groupBy("Survey ResponseID")
    .agg(
        spark_sum("Total Amount").alias("Total Spend")
    )
)

average_spend = customer_spending.agg(
    avg("Total Spend").alias("Average Spend per Customer")
)


# Convert to Pandas for visualisation
average_spend_pd = average_spend.toPandas()

# Plot
plt.figure(figsize=(7, 5))

ax = sns.barplot(
    data=average_spend_pd,
    x=["Average Spend"],
    y="Average Spend per Customer"
)

ax.bar_label(
    ax.containers[0],
    labels=[
        f"${average_spend_pd['Average Spend per Customer'].iloc[0]:,.2f}"
    ],
    padding=3
)

plt.title("Average Spend per Customer")
plt.xlabel("")
plt.ylabel("Average Spend ($)")
plt.tight_layout()

plt.show()

Analyse the Repeat Purchase Behavior of Customers

In [ ]:
## Repeat Purchase Analysis Behavior Per Customers
from pyspark.sql.functions import count, when, col

customer_purchase_counts = (
    cleaned_data
    .groupBy("Survey ResponseID")
    .agg(count("*").alias("Purchase Count"))
)

repeat_purchase_behavior = (
    customer_purchase_counts
    .withColumn(
        "Customer Type",
        when(col("Purchase Count") > 1, "Repeat Customer")
        .otherwise("One-Time Customer")
    )
    .groupBy("Customer Type")
    .count()
)

# Convert to Pandas for visualisation
repeat_purchase_pd = repeat_purchase_behavior.toPandas()

# Plot
plt.figure(figsize=(7, 5))

sns.barplot(
    data=repeat_purchase_pd,
    x="Customer Type",
    y="count"
)

plt.title("Repeat Purchase Behavior of Customers")
plt.xlabel("Customer Type")
plt.ylabel("Number of Customers")
plt.tight_layout()
plt.show()

Analyse the top 10 high-engagement customers

In [ ]:
# Top 10 High-Engagement Customers
from pyspark.sql.functions import count, col

high_engagement_customers = (
    cleaned_data
    .groupBy("Survey ResponseID")
    .agg(count("*").alias("Purchase Count"))
    .orderBy(col("Purchase Count").desc())
    .limit(10)
)

# Convert to Pandas for visualisation
high_engagement_pd = high_engagement_customers.toPandas()

# Plot
plt.figure(figsize=(12, 6))

sns.barplot(
    data=high_engagement_pd,
    x="Purchase Count",
    y="Survey ResponseID"
)

plt.title("Top 10 High-Engagement Customers")
plt.xlabel("Number of Purchases")
plt.ylabel("Survey Response ID")
plt.tight_layout()
plt.show()

## 3.10 Seasonal trends in product purchases and their impact on revenues <font color = red>[5 marks]</font> <br>

Investigate the seasonal trends in product purchases and their impact on the overall revenue.

In [ ]:
# Seasonal Trends in Product Purchases and Their Impact on Revenue

from pyspark.sql.functions import year, month, sum as spark_sum, col

# Extract year and month
seasonal_data = (
    cleaned_data
    .withColumn("Year", year(col("Order Date")))
    .withColumn("Month", month(col("Order Date")))
)


# Group by year and month, summing total revenue
seasonal_revenue = (
    seasonal_data
    .groupBy("Year", "Month")
    .agg(spark_sum("Total Amount").alias("Total Revenue"))
    .orderBy("Year", "Month")
)

# Convert to Pandas for visualisation
seasonal_revenue_pd = seasonal_revenue.toPandas()

# Plot
plt.figure(figsize=(12, 6))

sns.lineplot(
    data=seasonal_revenue_pd,
    x="Month",
    y="Total Revenue",
    hue="Year",
    marker="o",
    palette="tab10"
)

plt.title("Seasonal Trends and Their Impact on Revenue")
plt.xlabel("Month")
plt.ylabel("Total Revenue")
plt.xticks(range(1, 13))
plt.tight_layout()
plt.show()

## 3.11 Customer location vs purchasing behavior <font color = red>[5 marks]</font> <br>

Examine the relationship between customer's location and their purchasing behaviors

In [ ]:
# Relationship Between Customer Location and Purchase Behavior
from pyspark.sql.functions import sum as spark_sum, col

# Group purchases by state and total spend
state_spending = (
    cleaned_data
    .groupBy("Shipping Address State")
    .agg(spark_sum("Total Amount").alias("Total Spend"))
    .orderBy(col("Total Spend").desc())
)

# Convert to Pandas for visualisation
state_spending_pd = state_spending.toPandas()

# Plot revenue by state
plt.figure(figsize=(12, 12))

sns.barplot(
    data=state_spending_pd,
    x="Total Spend",
    y="Shipping Address State"
)

plt.title("Customer Location vs Purchasing Behavior")
plt.xlabel("Total Revenue")
plt.ylabel("Shipping State")
plt.tight_layout()
plt.show()

#4. Customer Segmentation and Insights <font color = red>[45 marks]</font> <br>


## 4.1 Perform RFM Analysis <font color = red>[10 marks]</font> <br>

RFM Analysis is a powerful customer segmentation technique used to evaluate and quantify customer value based on three key dimensions:
- **Recency**,
- **Frequency**,
- **Monetary**.

This method is particularly effective in identifying high-value customers, optimizing marketing strategies, and improving customer retention in the e-commerce industry.


### 1. Recency (R)
Recency measures how recently a customer made a purchase Customers who have purchased more recently are more likely to respond to promotions and offers.
- **Application:** By ranking customers based on the number of days since their last transaction, you can prioritize those who are most engaged.

### 2. Frequency (F)
Frequency counts the number of purchases a customer has made over a given period.
Frequent purchasers tend to be more loyal and are often a source of recurring revenue.
- **Application:** Analyzing purchase frequency helps in identifying consistent buyers and understanding their buying patterns.

### 3. Monetary (M)
Monetary value represents the total amount of money a customer has spent.
Customers who spend more are often more profitable, making them ideal targets for retention and upsell strategies.
- **Application:** By assessing the monetary contribution, you can distinguish between high-value and low-value customers.


### Prepare data for RFM Analysis <font color = red>[2 marks]</font> <br>


In [ ]:
from pyspark.sql.functions import datediff, max, count, sum, lit, col

# Get the latest order date in the dataset
latest_order_date = (
    cleaned_data
    .select(max("Order Date").alias("Latest Order Date"))
    .collect()[0]["Latest Order Date"]
)

# Calculate RFM metrics
rfm_data = (
    cleaned_data
    .groupBy("Survey ResponseID")
    .agg(
        datediff(
            lit(latest_order_date),
            max("Order Date")
        ).alias("Recency"),
        count("*").alias("Frequency"),
        sum("Total Amount").alias("Monetary")
    )
)

# Filter out customers with no purchases
rfm_data = rfm_data.filter(
    (col("Frequency") > 0) &
    (col("Monetary") > 0)
)


# Show RFM data
#rfm_data.show(10, truncate=False)
# Show RFM data in a formatted table
rfm_sample = rfm_data.limit(10)

In [ ]:
rfm_sample

In [ ]:
sc.install_pypi_package("scikit-learn")

In [ ]:
from pyspark.sql.functions import log1p, col
from sklearn.preprocessing import StandardScaler
# Apply log transformation to skewed features
rfm_data = (
    rfm_data
    .withColumn("Recency_log", log1p(col("Recency")))
    .withColumn("Monetary_log", log1p(col("Monetary")))
)


# Convert to Pandas DataFrame (for scikit-learn compatibility)
rfm_pd = rfm_data.toPandas()

# Scale features using StandardScaler
from sklearn.preprocessing import StandardScaler
scaler = StandardScaler()

rfm_scaled = scaler.fit_transform(
    rfm_pd[["Recency_log", "Frequency", "Monetary_log"]]
)

In [ ]:
from sklearn.cluster import KMeans

# Calculate the Within-Cluster Sum of Squares (WCSS)
wcss = []

for k in range(1, 11):
    kmeans = KMeans(
        n_clusters=k,
        random_state=42,
        n_init=10
    )
    kmeans.fit(rfm_scaled)
    wcss.append(kmeans.inertia_)

# Plot the elbow curve with the number of clusters on the x-axis and WCSS on the y-axis
plt.figure(figsize=(8, 5))
plt.plot(range(1, 11), wcss, marker="o")

plt.title("Elbow Method for Optimal Number of Clusters")
plt.xlabel("Number of Clusters")
plt.ylabel("WCSS")
plt.xticks(range(1, 11))
plt.tight_layout()
plt.show()

In [ ]:
# Fit the K-Means model using the optimal number of clusters obtained after understanding the elblow plot
optimal_k = 4  # Set this to the value identified from the elbow plot

kmeans = KMeans(
    n_clusters=optimal_k,
    random_state=42,
    n_init=10
)

# Add the assigned cluster labels to the Pandas DataFrame and convert back to PySpark if needed
rfm_pd["Cluster"] = kmeans.fit_predict(rfm_scaled).astype(int)

# Convert back to PySpark
rfm_clustered = spark.createDataFrame(rfm_pd)

rfm_clustered_sample = rfm_clustered.limit(10)

In [ ]:
rfm_clustered_sample

In [ ]:
# Convert the full RFM dataset from PySpark DataFrame to Pandas DataFrame for visualisation
rfm_visualisation_pd = rfm_clustered.toPandas()

# Generate a pairplot to visualise the relationships between the numeric RFM columns
sns.pairplot(
    rfm_visualisation_pd,
    vars=["Recency", "Frequency", "Monetary"],
    hue="Cluster",
    palette="tab10"
)

plt.show()

### Behavioral Trends Analysis <font color = red>[8 marks]</font> <br>

Perform RFM analysis to study the behavior of customers to tailor marketing strategies

In [ ]:
# Import necessary PySpark functions for data processing
from pyspark.sql.functions import datediff, max, countDistinct, sum, lit, log1p, col

# Group the dataset by 'Survey ResponseID' to calculate RFM (Recency, Frequency, Monetary) metrics
behavior_rfm = cleaned_data.groupBy("Survey ResponseID").agg(

# Compute 'Recency' as the difference between the latest date and the most recent order date
 datediff(
        lit(latest_order_date),
        max("Order Date")
    ).alias("Recency"),

# Compute 'Frequency' as the count of distinct product purchases (ASIN/ISBN)
countDistinct("ASIN/ISBN (Product Code)").alias("Frequency"),

# Compute 'Monetary' as the total spending sum for each customer
sum("Total Amount").alias("Monetary")
)

# Rename columns if required and normlise the distributions
behavior_rfm = (
    behavior_rfm
    .withColumn("Recency_log", log1p(col("Recency")))
    .withColumn("Monetary_log", log1p(col("Monetary")))
)

# Convert the processed RFM dataset back to Pandas for sklearn compatibility for clustering
behavior_rfm_pd = behavior_rfm.toPandas()

In [ ]:
# Apply K-Means clustering
behavior_scaled = scaler.fit_transform(
    behavior_rfm_pd[
        ["Recency_log", "Frequency", "Monetary_log"]
    ]
)

# Fit the K-Means model and predict cluster labels for each customer
behavior_kmeans = KMeans(
    n_clusters=4,
    random_state=42,
    n_init=10
)

cluster_labels = behavior_kmeans.fit_predict(behavior_scaled)

# Add the predicted cluster labels to the Pandas DataFrame
behavior_rfm_pd["Cluster"] = cluster_labels.astype(int)

# Convert the Pandas DataFrame back to a PySpark DataFrame
behavior_rfm_clustered = spark.createDataFrame(behavior_rfm_pd)

Analyse the Cluster Distribution by Income <font color = red>[2 marks]</font> <br>


In [ ]:
#Trend 1: Cluster Distribution by Income

# Import the necessary function for counting records in PySpark
from pyspark.sql.functions import count

# Join the RFM dataset with the survey dataset using a common key
cluster_income_data = behavior_rfm_clustered.join(
    survey.select("Survey ResponseID", "Q-demos-income"),
    on="Survey ResponseID",
    how="left"
)


# Aggregate data to count the number of customers per Cluster-Income group
cluster_income_counts = (
    cluster_income_data
    .groupBy("Cluster", "Q-demos-income")
    .agg(count("*").alias("Customer Count"))
)

# Convert the aggregated data from PySpark DataFrame to Pandas DataFrame for visualisation
cluster_income_pd = cluster_income_counts.toPandas()

# Plot
plt.figure(figsize=(12, 6))

sns.barplot(
    data=cluster_income_pd,
    x="Q-demos-income",
    y="Customer Count",
    hue="Cluster"
)

plt.title("Cluster Distribution by Income")
plt.xlabel("Income Group")
plt.ylabel("Number of Customers")
plt.xticks(rotation=45)
plt.tight_layout()
plt.show()

Analyse the Average Spending by Cluster <font color = red>[2 marks]</font> <br>


In [ ]:
#Trend 2: Average Spending by Cluster

# Import the required function for calculating averages in PySpark
from pyspark.sql.functions import avg

# Compute the average values of 'Recency_log', 'Frequency', and 'Monetary_log' for each customer cluster
cluster_summary = (
    behavior_rfm_clustered
    .groupBy("Cluster")
    .agg(
        avg("Recency_log").alias("Average Recency"),
        avg("Frequency").alias("Average Frequency"),
        avg("Monetary_log").alias("Average Monetary")
    )
    .orderBy("Cluster")
)

# Convert the aggregated cluster summary from PySpark DataFrame to Pandas DataFrame for visualisation
cluster_summary_pd = cluster_summary.toPandas()

# Generate a bar plot to visualise the average monetary spending per cluster
plt.figure(figsize=(8, 5))

sns.barplot(
    data=cluster_summary_pd,
    x="Cluster",
    y="Average Monetary"
)

plt.title("Average Spending by Cluster")
plt.xlabel("Cluster")
plt.ylabel("Average Monetary Spending (Log)")
plt.tight_layout()
plt.show()

Analyse the Purchase Frequency vs. Recency <font color = red>[2 marks]</font> <br>


In [ ]:
#Trend 3: Purchase Frequency vs. Recency

# Convert the RFM dataset from PySpark DataFrame to Pandas DataFrame for visualisation
frequency_recency_pd = behavior_rfm_clustered.toPandas()


# Generate a scatter plot to analyse the relationship between Purchase Frequency and Recency
plt.figure(figsize=(10, 6))

sns.scatterplot(
    data=frequency_recency_pd,
    x="Recency",
    y="Frequency",
    hue="Cluster",
    palette="tab10",
    alpha=0.7
)

plt.title("Purchase Frequency vs. Recency")
plt.xlabel("Recency (Days)")
plt.ylabel("Purchase Frequency")
plt.tight_layout()
plt.show()

Analyse the top categories by clusters <font color = red>[2 marks]</font> <br>


In [ ]:
#Trend 4: Top Categories by Cluster

# Import the necessary function to calculate the sum in PySpark
from pyspark.sql.functions import sum, col, row_number
from pyspark.sql.window import Window

# Join the merged dataset with the RFM dataset to associate each customer with their respective cluster
cluster_category_data = cleaned_data.join(
    behavior_rfm_clustered.select(
        "Survey ResponseID",
        "Cluster"
    ),
    on="Survey ResponseID",
    how="inner"
)

# Group the filtered data by 'Category' and compute the total spending in each category
category_spending = (
    cluster_category_data
    .groupBy("Cluster", "Category")
    .agg(sum("Total Amount").alias("Total Spending"))
)

# Order the categories by total spending in descending order and select the top 5 highest spending categories
cluster_window = Window.partitionBy("Cluster").orderBy(
    col("Total Spending").desc()
)

top_categories = (
    category_spending
    .withColumn("Rank", row_number().over(cluster_window))
    .filter(col("Rank") <= 5)
)

# Convert the top categories dataset from PySpark DataFrame to Pandas DataFrame for visualisation
top_categories_pd = top_categories.toPandas()

top_categories_pd = top_categories_pd.sort_values(
    ["Cluster", "Rank"]
)


# Plot the cluster
chart = sns.catplot(
    data=top_categories_pd,
    kind="bar",
    x="Total Spending",
    y="Category",
    col="Cluster",
    col_wrap=2,
    sharex=False,
    sharey=False,
    height=5,
    aspect=1.4
)

chart.set_titles("Cluster {col_name}")
chart.set_axis_labels(
    "Total Spending",
    "Category"
)

for axis in chart.axes.flat:
    axis.margins(x=0.25)

    for container in axis.containers:
        axis.bar_label(
            container,
            fmt="%.0f",
            padding=3
        )

chart.fig.suptitle(
    "Top 5 Categories by Cluster",
    y=1.03
)

plt.show()

## 4.2 Insights <font color = red>[35 marks]</font> <br>


### 4.2.1 When to schedule effective promotions. <font color = red>[3 marks]</font> <br>

Compare sales across weekdays to schedule effective promotions

In [ ]:
#Compare sales across weekdays to schedule effective promotions

from pyspark.sql.functions import dayofweek, sum, col

# Extract day of the week (1 = Sunday, 7 = Saturday)
weekday_sales_data = cleaned_data.withColumn(
    "Weekday",
    dayofweek(col("Order Date"))
)


# Group by weekday and sum total sales
weekday_sales = (
    weekday_sales_data
    .groupBy("Weekday")
    .agg(sum("Total Amount").alias("Total Sales"))
    .orderBy("Weekday")
)

# Convert to Pandas for visualisation
weekday_sales_pd = weekday_sales.toPandas()

weekday_names = {
    1: "Sunday",
    2: "Monday",
    3: "Tuesday",
    4: "Wednesday",
    5: "Thursday",
    6: "Friday",
    7: "Saturday"
}

weekday_sales_pd["Weekday Name"] = weekday_sales_pd["Weekday"].map(
    weekday_names
)

# Plot sales by weekday
plt.figure(figsize=(10, 6))

sns.barplot(
    data=weekday_sales_pd,
    x="Weekday Name",
    y="Total Sales"
)

plt.title("Total Sales Across Weekdays")
plt.xlabel("Weekday")
plt.ylabel("Total Sales")
plt.xticks(rotation=45)
plt.tight_layout()
plt.show()

### 4.2.2 Top-selling Products <font color = red>[2 marks]</font> <br>

Identify top-selling products by considering revenue and engagement metrics

In [ ]:
#Identify top-selling products using revenue and engagement metrics
from pyspark.sql.functions import sum, countDistinct, col

# Group by product and sum revenue
product_performance = (
    cleaned_data
    .groupBy("ASIN/ISBN (Product Code)")
    .agg(
        sum("Total Amount").alias("Total Revenue"),
        countDistinct("Survey ResponseID").alias("Customer Engagement")
    )
)

# Get top 10 products by revenue
top_selling_products = (
    product_performance
    .orderBy(col("Total Revenue").desc())
    .limit(10)
)

# Convert to Pandas for visualisation
top_selling_products_pd = top_selling_products.toPandas()

# Plot top products by revenue
plt.figure(figsize=(12, 7))

sns.barplot(
    data=top_selling_products_pd,
    x="Total Revenue",
    y="ASIN/ISBN (Product Code)"
)

plt.title("Top 10 Products by Revenue")
plt.xlabel("Total Revenue")
plt.ylabel("Product Title")
plt.tight_layout()
plt.show()

### 4.2.3 State-wise revenue Distribution <font color = red>[5 marks]</font> <br>

Assess state-wise revenue to focus on high-growth areas

In [ ]:
#Assess state-wise revenue to focus on high-growth areas
from pyspark.sql.functions import sum, col

# Group by state and sum revenue
state_revenue = (
    cleaned_data
    .groupBy("Shipping Address State")
    .agg(sum("Total Amount").alias("Total Revenue"))
    .orderBy(col("Total Revenue").desc())
)

# Convert to Pandas for visualisation
state_revenue_pd = state_revenue.toPandas()

# Plot revenue by state
plt.figure(figsize=(12, 12))

sns.barplot(
    data=state_revenue_pd,
    x="Total Revenue",
    y="Shipping Address State"
)

plt.title("State-Wise Revenue Distribution")
plt.xlabel("Total Revenue")
plt.ylabel("Shipping State")
plt.tight_layout()
plt.show()

### 4.2.4 Repeat Purchase Behavior <font color = red>[5 marks]</font> <br>

Examine repeat purchase behavior to enhance retention initiatives.

In [ ]:
#Examine repeat purchase behavior to enhance retention initiatives

from pyspark.sql.functions import count, col

# Count total purchases per customer
customer_purchase_counts = (
    cleaned_data
    .groupBy("Survey ResponseID")
    .agg(count("*").alias("Total Purchases"))
)

# Filter for repeat customers (those with more than one purchase)
repeat_customers = customer_purchase_counts.filter(
    col("Total Purchases") > 1
)

# Show sample data
repeat_customers.show(10, truncate=False)

### 4.2.5 Flagging Potential Fraud <font color = red>[5 marks]</font> <br>

Identify irregular transaction patterns to flag potential fraud.

In [ ]:
#Identify irregular transaction patterns to flag potential fraud

from pyspark.sql.functions import col, avg, stddev

# Calculate the threshold for unusually high spending
spending_statistics = cleaned_data.agg(
    avg("Total Amount").alias("Mean Spending"),
    stddev("Total Amount").alias("Spending Standard Deviation")
).collect()[0]

mean_spending = spending_statistics["Mean Spending"]
spending_stddev = spending_statistics["Spending Standard Deviation"]

# Consider spending to be unusually high if the total spent is greater than the mean + 3 * std dev
fraud_threshold = mean_spending + (3 * spending_stddev)

# Filter transactions that exceed the threshold
suspicious_transactions = cleaned_data.filter(
    col("Total Amount") > fraud_threshold
)

# Show suspicious transactions
#suspicious_transactions.show(10, truncate=False)
suspicious_transactions_sample = suspicious_transactions.limit(5)

In [ ]:
suspicious_transactions_sample

### 4.2.6 Demand Variations across product categories <font color = red>[5 marks]</font> <br>

Perform inventory management by monitoring demand variations across product categories.

In [ ]:
#Monitor demand variations across product categories (Top 25) for inventory management

from pyspark.sql.functions import col, sum
import matplotlib.pyplot as plt
import seaborn as sns

# Group by category and month, summing total revenue
category_trends = (
    cleaned_data
    .groupBy("Category", "Order Month")
    .agg(sum("Total Amount").alias("Monthly Revenue"))
)

# Compute total revenue per category
category_revenue = (
    category_trends
    .groupBy("Category")
    .agg(sum("Monthly Revenue").alias("Total Revenue"))
)

# Get the top 25 categories by total revenue
top_25_categories = (
    category_revenue
    .orderBy(col("Total Revenue").desc())
    .limit(25)
    .select("Category")
)

# Filter category_trends to include only top 25 categories
top_category_trends = category_trends.join(
    top_25_categories,
    on="Category",
    how="inner"
)

# Convert to Pandas for visualisation
top_category_trends_pd = top_category_trends.toPandas()

# Plot revenue trends for top 25 categories
plt.figure(figsize=(16, 9))

sns.lineplot(
    data=top_category_trends_pd,
    x="Order Month",
    y="Monthly Revenue",
    hue="Category",
    marker="o"
)

plt.title("Monthly Revenue Trends for Top 25 Product Categories")
plt.xlabel("Month")
plt.ylabel("Total Revenue")
plt.xticks(range(1, 13))
plt.legend(
    title="Category",
    bbox_to_anchor=(1.05, 1),
    loc="upper left"
)
plt.tight_layout()
plt.show()

### 4.2.7 Assess how bulk purchases affect revenue and supply chain operations <font color = red>[5 marks]</font> <br>

Analyse the impact of how bulk purchasing behavior affects revenue and the overall supply chain operations.

In [ ]:
#Assess how bulk purchases affect revenue and supply chain operations
from pyspark.sql.functions import col, sum

# Filter bulk purchases (Quantity > 5) and compute total revenue per category
bulk_purchase_revenue = (
    cleaned_data
    .filter(
        (col("Quantity") > 5) &
        col("ASIN/ISBN (Product Code)").isNotNull() &
        (col("ASIN/ISBN (Product Code)") != "Unknown")
    )
    .groupBy("ASIN/ISBN (Product Code)")
    .agg(
        sum("Total Amount").alias("Bulk Purchase Revenue"),
        sum("Quantity").alias("Bulk Units Purchased")
    )
)

# Select the top 25 categories by total revenue
top_bulk_products = (
    bulk_purchase_revenue
    .orderBy(col("Bulk Purchase Revenue").desc())
    .limit(25)
)

# Convert to Pandas for visualisation
top_bulk_products_pd = top_bulk_products.toPandas()

# Plot revenue from bulk purchases (Top 25 categories)
plt.figure(figsize=(12, 8))

sns.barplot(
    data=top_bulk_products_pd,
    x="Bulk Purchase Revenue",
    y="ASIN/ISBN (Product Code)"
)

plt.title("Revenue from Bulk Purchases by Product")
plt.xlabel("Bulk Purchase Revenue")
plt.ylabel("Product Code")
plt.tight_layout()

plt.show()

### 4.2.8 Compare lifecycle strategies <font color = red>[5 marks]</font> <br>

Compare new and established products to inform and compare lifecycle strategies to make informed decisions.

In [ ]:
#Compare new and established products to inform lifecycle strategies

from pyspark.sql.functions import min, sum

# Compute "Launch Year" as the first recorded sale year for each product
product_launch_year = (
    cleaned_data
    .groupBy("ASIN/ISBN (Product Code)")
    .agg(min("Order Year").alias("Launch Year"))
)

# Join this back to the main dataset
product_lifecycle_data = cleaned_data.join(
    product_launch_year,
    on="ASIN/ISBN (Product Code)",
    how="left"
)

# Now, we can compute revenue by launch year
revenue_by_launch_year = (
    product_lifecycle_data
    .groupBy("Launch Year")
    .agg(sum("Total Amount").alias("Total Revenue"))
    .orderBy("Launch Year")
)

# Convert to Pandas
revenue_by_launch_year_pd = revenue_by_launch_year.toPandas()

# Plot revenue vs. launch year
plt.figure(figsize=(10, 6))

sns.barplot(
    data=revenue_by_launch_year_pd,
    x="Launch Year",
    y="Total Revenue"
)

plt.title("Revenue by Product Launch Year")
plt.xlabel("Launch Year")
plt.ylabel("Total Revenue")
plt.tight_layout()
plt.show()

#5 Conclusion <font color = red>[10 marks]</font> <br>

Write your conclusion.